# **Wikipedia Article Scraper**

In [ ]:
# Step 1: imports
import requests
from bs4 import BeautifulSoup

In [ ]:
# Step 2: fetch page HTML
def get_wikipedia_page(topic):
    url = f"https://en.wikipedia.org/wiki/{topic.replace(' ', '_')}"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch page (status {response.status_code})")
        return None

In [ ]:
# Step 3: parse title
def get_article_title(soup):
    title_tag = soup.find("h1", id="firstHeading")
    return title_tag.text.strip() if title_tag else "No title found"

In [ ]:
# Step 4: parse summary
def get_article_summary(soup):
    for p in soup.select("p"):
        text = p.get_text(strip=True)
        if len(text) > 100:         # threshold to avoid empty/too-short paragraphs
            return text
    return "No summary found."

In [ ]:
# Step 5: headings and related links
def get_headings(soup):
    headings = []
    for tag in soup.find_all(["h2", "h3", "h4"]):
        text = tag.get_text(" ", strip=True).replace("[edit]", "")
        headings.append(text)
    return headings

def get_related_links(soup, limit=5):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/wiki/") and ":" not in href:
            links.append(f"https://en.wikipedia.org{href}")
    return list(dict.fromkeys(links))[:limit]   # preserve order, remove duplicates

In [ ]:
# Step 6: print and save
def save_to_file(title, summary, headings, links, filename="wikipedia_result.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("--- Wikipedia Article Details ---\n\n")
        f.write(f"Title: {title}\n\n")
        f.write(f"Summary:\n{summary}\n\n")
        f.write("Headings:\n")
        for h in headings:
            f.write(f"- {h}\n")
        f.write("\nRelated Links:\n")
        for link in links:
            f.write(f"- {link}\n")
    print(f"Saved to '{filename}'")

In [ ]:
# Step 7: Main Program

def main():
    topic = input("Enter Wikipedia topic: ").strip()
    html = get_wikipedia_page(topic)
    if not html:
        return

    soup = BeautifulSoup(html, "html.parser")
    title = get_article_title(soup)
    summary = get_article_summary(soup)
    headings = get_headings(soup)
    links = get_related_links(soup)

    print("\n--- Results ---")
    print("Title:", title)
    print("\nSummary:\n", summary)
    print("\nHeadings:")
    for h in headings[:8]:
        print("-", h)
    print("\nRelated Links:")
    for l in links:
        print("-", l)

    save_to_file(title, summary, headings, links)

if __name__ == "__main__":
    main()

Enter Wikipedia topic: Artificial intelligence

--- Results ---
Title: Artificial intelligence

Summary:
 Artificial intelligence(AI) is the capability ofcomputational systemsto perform tasks typically associated withhuman intelligence, such aslearning,reasoning,problem-solving,perception, anddecision-making. It is afield of researchincomputer sciencethat develops and studies methods andsoftwarethat enable machines toperceive their environmentand uselearningandintelligenceto take actions that maximize their chances of achieving defined goals.[1]

Headings:
- Contents
- Goals
- Reasoning and problem-solving
- Knowledge representation
- Planning and decision-making
- Learning
- Natural language processing
- Perception

Related Links:
- https://en.wikipedia.org/wiki/Main_Page
- https://en.wikipedia.org/wiki/Artificial_intelligence
- https://en.wikipedia.org/wiki/AI_(disambiguation)
- https://en.wikipedia.org/wiki/Artificial_intelligence_(disambiguation)
- https://en.wikipedia.org/wiki/Art